# CS 598 DL4H — WatchSleepNet Colab POC

This notebook validates the Colab ↔ GitHub ↔ Google Drive workflow for the WatchSleepNet replication project.

**Branch:** `colab-poc`  
**Repo:** [cs598-DLH-WatchSleepNet](https://github.com/jananij2/cs598-DLH-WatchSleepNet)

## 1. Install Dependencies

In [ ]:
# Install PyHealth from source (use this during active contribution work)
!pip install git+https://github.com/sunlabuiuc/PyHealth.git -q

# Other common dependencies
!pip install numpy pandas scikit-learn matplotlib -q

In [ ]:
import pyhealth
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"PyHealth version: {pyhealth.__version__}")
print("All imports successful.")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Base path — adjust if your Drive folder structure differs
DRIVE_BASE = '/content/drive/MyDrive/cs598-watchsleepnet'

PATHS = {
    'raw':       os.path.join(DRIVE_BASE, 'raw'),
    'processed': os.path.join(DRIVE_BASE, 'processed'),
    'models':    os.path.join(DRIVE_BASE, 'models'),
    'results':   os.path.join(DRIVE_BASE, 'results'),
}

for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"{name:12s} → {path}")

## 3. Drive POC — Generate, Save, Reload, and Plot

In [ ]:
# --- Generate synthetic IBI data (stand-in for real DREAMT data) ---
rng = np.random.default_rng(seed=42)

n_samples = 500
ibi_ms = 800 + rng.normal(0, 50, n_samples)   # ~800 ms mean (75 bpm)
timestamps = np.cumsum(ibi_ms) / 1000.0        # seconds
# Fake sleep stage labels: 0=Wake, 1=NREM, 2=REM
labels = rng.integers(0, 3, n_samples)

df_raw = pd.DataFrame({'timestamp_s': timestamps, 'ibi_ms': ibi_ms, 'label': labels})
print(f"Generated {len(df_raw)} synthetic IBI samples")
df_raw.head()

In [ ]:
# --- Save raw synthetic data to Drive ---
raw_path = os.path.join(PATHS['raw'], 'synthetic_ibi.csv')
df_raw.to_csv(raw_path, index=False)
print(f"Saved raw data → {raw_path}")

In [ ]:
# --- Reload from Drive and post-process ---
df_loaded = pd.read_csv(raw_path)
print(f"Loaded {len(df_loaded)} rows from Drive")

# Normalize IBI (z-score)
df_loaded['ibi_normalized'] = (
    (df_loaded['ibi_ms'] - df_loaded['ibi_ms'].mean()) / df_loaded['ibi_ms'].std()
)

# Rolling mean (simple smoothing)
df_loaded['ibi_smooth'] = df_loaded['ibi_ms'].rolling(window=10, center=True).mean()

df_loaded.head()

In [ ]:
# --- Save processed data to Drive ---
processed_path = os.path.join(PATHS['processed'], 'synthetic_ibi_processed.csv')
df_loaded.to_csv(processed_path, index=False)
print(f"Saved processed data → {processed_path}")

In [ ]:
# --- Plot: raw vs smoothed IBI colored by sleep stage ---
stage_colors = {0: 'steelblue', 1: 'seagreen', 2: 'tomato'}
stage_labels = {0: 'Wake', 1: 'NREM', 2: 'REM'}

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for stage, color in stage_colors.items():
    mask = df_loaded['label'] == stage
    axes[0].scatter(
        df_loaded.loc[mask, 'timestamp_s'],
        df_loaded.loc[mask, 'ibi_ms'],
        s=4, color=color, label=stage_labels[stage], alpha=0.7
    )

axes[0].plot(df_loaded['timestamp_s'], df_loaded['ibi_smooth'], color='black', lw=1.5, label='Smoothed')
axes[0].set_ylabel('IBI (ms)')
axes[0].set_title('Synthetic IBI Signal by Sleep Stage')
axes[0].legend(loc='upper right', markerscale=3)

axes[1].plot(df_loaded['timestamp_s'], df_loaded['ibi_normalized'], lw=0.8, color='slategray')
axes[1].set_ylabel('IBI (z-score)')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Normalized IBI')

plt.tight_layout()

# Save figure to Drive
fig_path = os.path.join(PATHS['results'], 'ibi_plot.png')
plt.savefig(fig_path, dpi=150)
print(f"Plot saved → {fig_path}")
plt.show()

## 4. PyHealth Smoke Test

Verify PyHealth is importable and explore the available modules relevant to our contribution.

In [ ]:
# Browse PyHealth's public API surface
import pyhealth.datasets as ph_datasets
import pyhealth.models as ph_models
import pyhealth.tasks as ph_tasks

print("=== Available Datasets ===")
print([x for x in dir(ph_datasets) if not x.startswith('_')])

print("\n=== Available Models ===")
print([x for x in dir(ph_models) if not x.startswith('_')])

print("\n=== Available Tasks ===")
print([x for x in dir(ph_tasks) if not x.startswith('_')])

## Next Steps

- [ ] Download DREAMT dataset from PhysioNet → `raw/`
- [ ] Implement IBI preprocessing pipeline → `processed/`
- [ ] Implement `WatchSleepNet` as `pyhealth.models.WatchSleepNet(BaseModel)`
- [ ] Wire up DREAMT dataset class as `pyhealth.datasets.DREAMTDataset(BaseDataset)`
- [ ] 5-fold CV training loop → save checkpoints to `models/`
- [ ] Compare against paper's reported metrics (Table 2)